# Gyroscope Debiasing using ARIMA (Driver E - VW01)

This notebook creates a baseline ARIMA model to estimate and remove the drift/bias from the raw gyroscope data. We will use `pmdarima` for hyperparameter tuning (`auto_arima`) to find the optimal (p,d,q) parameters.

In [1]:
# Install required libraries if you don't have them
%pip install pandas numpy matplotlib statsmodels pmdarima


[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
import pmdarima as pm

import warnings
warnings.filterwarnings('ignore')

### 1. Load the Smartphone Dataset

In [4]:
file_path = '/Users/hamza/SIH/IO-VNBD/Synchronised V abd S datasets/Categorised IOVNB Dataset/Vw (Driver E)/Vw01/S-Vw1.csv'

# Load data with latin-1 encoding
df = pd.read_csv(file_path, encoding='latin-1')
# Strip any leading/trailing spaces from column names
df.columns = df.columns.str.strip()

# Extract time (convert ms to seconds) and Gyroscope columns
time_sec = df['TIME SINCE START (ms)'] / 1000.0
gyro_yaw = df['GYROSCOPE Yaw (rad/s)']
gyro_pitch = df['GYROSCOPE Pitch (rad/s)']
gyro_roll = df['GYROSCOPE Roll (rad/s)']

# For hyperparameter tuning, we will use a subset of the data (e.g., first 1000 points) to speed up training
# ARIMA can be slow on very large datasets (~90k points)
subset_size = 1000
train_yaw = gyro_yaw.iloc[:subset_size]
time_subset = time_sec.iloc[:subset_size]

plt.figure(figsize=(12, 4))
plt.plot(time_subset, train_yaw, label='Raw Gyro Yaw')
plt.title('Raw Gyroscope Yaw (First 1000 points)')
plt.xlabel('Time (s)')
plt.ylabel('Yaw (rad/s)')
plt.legend()
plt.show()

KeyError: 'TIME SINCE START (ms)'

### 2. Hyperparameter Tuning using Auto-ARIMA
We use `pmdarima.auto_arima` to search for the best `(p, d, q)` parameters. By default, you requested ARIMA(1,1,1) as a baseline, but `auto_arima` will verify if this is optimal by minimizing the AIC (Akaike Information Criterion) score.

In [ ]:
print("Starting Auto-ARIMA tuning for Gyro Yaw...")

# Run auto_arima to find best hyperparameters
# We constrain d=1 to handle the drift/random walk nature of gyro bias
auto_model = pm.auto_arima(
    train_yaw, 
    start_p=0, start_q=0,
    max_p=3, max_q=3, 
    d=1,                # Enforce 1 level of differencing as requested
    seasonal=False,     # Gyro drift is generally non-seasonal over short periods
    trace=True,
    error_action='ignore',  
    suppress_warnings=True, 
    stepwise=True
)

print("\nBest Model Summary:")
print(auto_model.summary())

### 3. Fit the Baseline Model & Debias
Once the best parameters are found (or using the requested baseline ARIMA(1,1,1)), we fit the model to estimate the underlying bias (trend). We then subtract this trend from the raw signal.

In [ ]:
# Get the optimal parameters found (or force (1,1,1))
best_order = auto_model.order
print(f"Using ARIMA order: {best_order}")

# You can uncomment the line below to strictly force ARIMA(1,1,1) if tuning suggests otherwise
# best_order = (1, 1, 1)

# Fit statsmodels ARIMA
model = ARIMA(train_yaw, order=best_order)
fitted_model = model.fit()

# The 'fittedvalues' represent the model's estimate of the signal + bias at each step.
# To debias, we subtract the rolling estimated bias. 
# In differenced models, the predicted trend is often the fitted values.
estimated_bias = fitted_model.predict(typ='levels')

# Debiased Signal = Raw - Estimated Bias
debiased_yaw = train_yaw - estimated_bias

plt.figure(figsize=(14, 6))
plt.plot(time_subset, train_yaw, label='Raw Gyro Yaw', alpha=0.5)
plt.plot(time_subset, estimated_bias, label='ARIMA Estimated Bias/Trend', color='red', linewidth=2)
plt.plot(time_subset, debiased_yaw, label='Debiased Gyro Yaw', color='green', alpha=0.8)

plt.title(f'Gyroscope Debiasing using ARIMA{best_order}')
plt.xlabel('Time (s)')
plt.ylabel('Yaw (rad/s)')
plt.legend()
plt.grid(True)
plt.show()